In [1]:
!pip install PySide6 pyproj ultralytics opencv-python numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.8/557.8 kB 5.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 MB 14.4 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 MB 25.6 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [PySide6]m3/4 [PySide6]Addons]als]


In [1]:
import sys
import os
import cv2
import math
import re
import numpy as np
import torch
from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout,
                               QHBoxLayout, QLabel, QPushButton, QComboBox,
                               QTextEdit, QFileDialog, QGroupBox, QLineEdit,
                               QFormLayout, QFrame, QGridLayout, QDoubleSpinBox)
from PySide6.QtGui import QPixmap, QImage, QPainter, QPen, QColor
from PySide6.QtCore import Qt, QRect, QThread, Signal

from ultralytics import YOLO
from pyproj import Geod, Proj, Transformer

YOLO_ONLY_WEIGHTS = "runs/detect/yolo_only/weights/best.pt"
YOLO_ZONE_WEIGHTS = "runs/detect/yolo_zone_v2/weights/best.pt"
LPRNET_WEIGHTS = "weights_v2/lprnet_best.pth"
LPRNET_REPO_PATH = 'LPRNet_Pytorch'

if LPRNET_REPO_PATH not in sys.path:
    sys.path.append(LPRNET_REPO_PATH)
try:
    from model.LPRNet import LPRNet
except ImportError:
    print("⚠️ LPRNet repo not found!")

CHARS = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '.']
YOLO_NAMES = {
    0: '0', 1: '1', 2: '2', 3: '3', 4: '4',
    5: '5', 6: '6', 7: '7', 8: '8', 9: '9',
    10: 'minus', 11: 'point', 12: 'tag'
}
CHAR_MAP = {
    '0': '0', '1': '1', '2': '2', '3': '3', '4': '4',
    '5': '5', '6': '6', '7': '7', '8': '8', '9': '9',
    'minus': '-', 'point': '.', 'tag': ''
}


def helmert_transform(x, y, z, params):
    """
    Виконує 7-параметричне перетворення.
    params: {dx, dy, dz, rx, ry, rz, s}
    rx, ry, rz - у кутових секундах
    s - у ppm (parts per million)
    """
    dx, dy, dz = params['dx'], params['dy'], params['dz']
    rx_sec, ry_sec, rz_sec = params['rx'], params['ry'], params['rz']
    s_ppm = params['s']

    # Конвертація одиниць
    # Кутові секунди в радіани: sec * (pi / 180 / 3600)
    rad_factor = math.pi / (180.0 * 3600.0)
    rx = rx_sec * rad_factor
    ry = ry_sec * rad_factor
    rz = rz_sec * rad_factor

    # Масштабний коефіцієнт (m = 1 + s * 10^-6)
    m = 1 + (s_ppm * 1e-6)

    # Вектор координат (Source)
    vec_src = np.array([[x], [y], [z]])

    # Вектор зсуву (Translation)
    vec_trans = np.array([[dx], [dy], [dz]])

    # Матриця обертання (для малих кутів)
    # R = [[ 1, -rz,  ry],
    #      [ rz,  1, -rx],
    #      [-ry, rx,   1]]
    rot_mat = np.array([
        [1.0, -rz, ry],
        [rz, 1.0, -rx],
        [-ry, rx, 1.0]
    ])

    # Формула: X_target = T + m * R * X_source
    vec_target = vec_trans + m * (rot_mat @ vec_src)

    return vec_target[0][0], vec_target[1][0], vec_target[2][0]


def ecef_to_lla(x, y, z):
    """
    Конвертує Geocentric XYZ (ECEF) -> Latitude, Longitude, Altitude (WGS84)
    """
    # Використовуємо pyproj для надійності
    transformer = Transformer.from_crs({"proj": 'geocent', "ellps": 'WGS84', "datum": 'WGS84'},
                                       {"proj": 'latlong', "ellps": 'WGS84', "datum": 'WGS84'},
                                       always_xy=True)
    lon, lat, alt = transformer.transform(x, y, z)
    return lat, lon, alt


# ==========================================
# 2. WORKER THREAD
# ==========================================
class LogicWorker(QThread):
    result_ready = Signal(object, str, str)

    def __init__(self, params):
        super().__init__()
        self.params = params
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def run(self):
        img_orig = self.params['image']
        roi = self.params['roi']
        mode = self.params['model_mode']

        # --- CROP ---
        x_off, y_off, w, h = roi
        if w > 0 and h > 0:
            img_process = img_orig[y_off:y_off + h, x_off:x_off + w]
        else:
            img_process = img_orig
            x_off, y_off = 0, 0

        detected_texts = []
        vis_img = img_process.copy()

        # --- RECOGNITION ---
        if mode == "YOLO Only":
            model = YOLO(YOLO_ONLY_WEIGHTS)
            res = model(img_process, conf=0.25, verbose=False)[0]
            chars = []
            for box in res.boxes:
                cls = int(box.cls[0]);
                sym = CHAR_MAP.get(YOLO_NAMES.get(cls, '?'), '')
                if not sym: continue
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                chars.append({'char': sym, 'cx': (x1 + x2) / 2, 'cy': (y1 + y2) / 2, 'x': x1, 'y': y1, 'w': x2 - x1,
                              'h': y2 - y1, 'conf': float(box.conf[0])})
                cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 165, 255), 1)
            detected_texts = self.organize_text_lines_chars(chars)

        elif mode == "YOLO + LPR":
            y_zone = YOLO(YOLO_ZONE_WEIGHTS)
            lpr = LPRNet(lpr_max_len=18, phase=False, class_num=13, dropout_rate=0).to(self.device)
            lpr.load_state_dict(torch.load(LPRNET_WEIGHTS, map_location=self.device))
            lpr.eval()

            res = y_zone(img_process, verbose=False)[0]
            boxes = self.organize_lpr_zones(res.boxes.xyxy.cpu().numpy().astype(int).tolist())

            for b in boxes:
                x1, y1, x2, y2 = b;
                x1, y1 = max(0, x1), max(0, y1);
                x2, y2 = min(img_process.shape[1], x2), min(img_process.shape[0], y2)
                crop = img_process[y1:y2, x1:x2]
                if crop.size == 0: continue
                blob = cv2.resize(crop, (94, 24)).astype('float32');
                blob -= 127.5;
                blob *= 0.0078125
                blob = np.transpose(blob, (2, 0, 1));
                blob = torch.from_numpy(blob).unsqueeze(0).to(self.device)
                with torch.no_grad():
                    txt = self.decode_lpr(lpr(blob))
                    detected_texts.append(txt)
                cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(vis_img, txt, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        # --- CALCULATION ---
        raw_text_combined = " ".join(detected_texts)
        gps_result = self.calculate_coords(raw_text_combined)

        # Merge
        final_vis = img_orig.copy()
        if w > 0 and h > 0:
            final_vis[y_off:y_off + h, x_off:x_off + w] = vis_img
            cv2.rectangle(final_vis, (x_off, y_off), (x_off + w, y_off + h), (255, 0, 255), 2)
        else:
            final_vis = vis_img

        self.result_ready.emit(final_vis, raw_text_combined, gps_result)

    # --- HELPERS (Same as before + LPR sort) ---
    def organize_text_lines_chars(self, chars):
        # (Вставити вашу функцію з попередніх скриптів для векторного сортування)
        # Для економії місця я використовую скорочену версію, але вставте туди повну
        if not chars: return []
        chars.sort(key=lambda c: c['conf'], reverse=True)
        valid = []
        for c in chars:
            if not any(math.hypot(c['cx'] - v['cx'], c['cy'] - v['cy']) < 5.0 for v in valid): valid.append(c)
        chars = sorted(valid, key=lambda c: c['cx'])
        lines = []
        for char in chars:
            best = None;
            mn = float('inf')
            for l in lines:
                last = l[-1];
                dx = char['cx'] - last['cx'];
                dy = char['cy'] - last['cy']
                if dx <= 0: continue
                dist = math.hypot(dx, dy);
                ang = math.degrees(math.atan2(dy, dx))
                if abs(ang) > 25 or dist > (last['w'] + char['w'] + last['h'] + char['h']) / 4 * 5: continue
                if dist < mn: mn = dist; best = l
            if best:
                best.append(char)
            else:
                lines.append([char])
        lines.sort(key=lambda l: sum(c['cy'] for c in l) / len(l))
        final = []
        for l in lines:
            s = "";
            px = l[0]['x'] + l[0]['w']
            for i, c in enumerate(l):
                if i > 0 and (c['x'] - px) > (c['w'] + l[i - 1]['w']) / 2 * 0.8: s += " "
                s += c['char'];
                px = c['x'] + c['w']
            final.append(s)
        return final

    def organize_lpr_zones(self, boxes):
        if not boxes: return []
        zones = [{'box': b, 'cy': (b[1] + b[3]) / 2, 'cx': (b[0] + b[2]) / 2, 'h': b[3] - b[1]} for b in boxes]
        zones.sort(key=lambda z: z['cy'])
        rows = []
        if zones:
            curr = [zones[0]]
            for i in range(1, len(zones)):
                c = zones[i];
                p = curr[-1]
                if abs(c['cy'] - p['cy']) < p['h'] * 0.6:
                    curr.append(c)
                else:
                    rows.append(curr); curr = [c]
            rows.append(curr)
        final = []
        for r in rows:
            r.sort(key=lambda z: z['cx'])
            for item in r: final.append(item['box'])
        return final

    def decode_lpr(self, preds):
        preds = preds.cpu().detach().numpy();
        labels = np.argmax(preds, axis=1);
        decoded = []
        for i in range(labels.shape[0]):
            line = labels[i, :];
            res = [];
            pre = -1
            for c in line:
                if c != pre and c != 12: res.append(CHARS[c])
                pre = c
            decoded.append("".join(res))
        return decoded[0]

    def calculate_coords(self, text):
        # Шукаємо всі числа (включно з від'ємними і дробовими)
        nums = re.findall(r'-?\d+\.?\d*', text)
        nums = [float(n) for n in nums]

        calc_mode = self.params['calc_mode']

        # 1. POLAR
        if calc_mode == "Polar (Az, Dist)":
            if len(nums) >= 2:
                az, dist = nums[0], nums[1]
                geod = Geod(ellps='WGS84')
                lon_end, lat_end, _ = geod.fwd(self.params['lon'], self.params['lat'], az, dist)
                return f"MODE: Polar\nAz={az}, Dist={dist}\n\nGPS: {lat_end:.6f}, {lon_end:.6f}"
            else:
                return "Error: Need 2 numbers (Az, Dist)"

        # 2. CARTESIAN (X, Y)
        elif calc_mode == "Cartesian (X, Y)":
            if len(nums) >= 2:
                x, y = nums[0], nums[1]
                p = Proj(proj='ortho', lat_0=self.params['lat'], lon_0=self.params['lon'], datum='WGS84')
                lon_end, lat_end = p(x, y, inverse=True)
                return f"MODE: Cartesian 2D\nX={x}, Y={y}\n\nGPS: {lat_end:.6f}, {lon_end:.6f}"
            else:
                return "Error: Need 2 numbers (X, Y)"

        # 3. HELMERT (X, Y, H -> GPS)
        elif calc_mode == "Helmert (X, Y, H)":
            if len(nums) >= 3:
                x, y, h = nums[0], nums[1], nums[2]
                hp = self.params['helmert']

                # Крок 1: Трансформація
                x_prime, y_prime, z_prime = helmert_transform(x, y, h, hp)

                # Крок 2: ECEF -> LLA
                lat, lon, alt = ecef_to_lla(x_prime, y_prime, z_prime)

                return (f"MODE: Helmert 7-Param\n"
                        f"Input Local: X={x}, Y={y}, H={h}\n"
                        f"Transformed ECEF: {x_prime:.2f}, {y_prime:.2f}, {z_prime:.2f}\n\n"
                        f"GPS RESULT:\nLat: {lat:.6f}\nLon: {lon:.6f}\nAlt: {alt:.2f} m")
            else:
                return "Error: Need 3 numbers (X, Y, H)"

        return "Unknown Mode"


# ==========================================
# 3. GUI
# ==========================================
class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Radar Digit Recognizer & Helmert Transformer")
        self.resize(1250, 850)
        self.current_cv_image = None

        self.setStyleSheet("""
            QMainWindow { background-color: #2b2b2b; color: white; }
            QLabel { color: #e0e0e0; }
            QGroupBox { border: 1px solid #555; margin-top: 10px; font-weight: bold; color: #00bcd4; }
            QGroupBox::title { subcontrol-origin: margin; subcontrol-position: top center; padding: 0 5px; }
            QPushButton { background-color: #3d3d3d; border: 1px solid #555; padding: 6px; border-radius: 4px; color: white; }
            QPushButton:hover { background-color: #505050; border-color: #00bcd4; }
            QLineEdit, QComboBox, QTextEdit, QDoubleSpinBox { background-color: #1e1e1e; border: 1px solid #555; color: white; padding: 4px; }
        """)

        main_widget = QWidget()
        self.setCentralWidget(main_widget)
        layout = QHBoxLayout(main_widget)

        # --- LEFT PANEL ---
        panel = QFrame();
        panel.setFixedWidth(400)
        vbox = QVBoxLayout(panel)

        # Image Load
        grp1 = QGroupBox("1. Зображення")
        l1 = QHBoxLayout()
        btn1 = QPushButton("📂 Файл");
        btn1.clicked.connect(self.load_img)
        btn2 = QPushButton("📋 Буфер");
        btn2.clicked.connect(self.paste_img)
        l1.addWidget(btn1);
        l1.addWidget(btn2)
        grp1.setLayout(l1)
        vbox.addWidget(grp1)

        # Radar Params (Simple)
        grp2 = QGroupBox("2. Позиція Радара (Origin)")
        f2 = QFormLayout()
        self.lat = QLineEdit("48.5");
        self.lon = QLineEdit("35.1")
        f2.addRow("Lat:", self.lat);
        f2.addRow("Lon:", self.lon)
        grp2.setLayout(f2)
        vbox.addWidget(grp2)

        # Mode Selection
        grp3 = QGroupBox("3. Режим")
        v3 = QVBoxLayout()
        self.cb_model = QComboBox();
        self.cb_model.addItems(["YOLO Only", "YOLO + LPR"])
        v3.addWidget(QLabel("Модель:"))
        v3.addWidget(self.cb_model)

        self.cb_mode = QComboBox()
        self.cb_mode.addItems(["Polar (Az, Dist)", "Cartesian (X, Y)", "Helmert (X, Y, H)"])
        self.cb_mode.currentIndexChanged.connect(self.toggle_helmert)
        v3.addWidget(QLabel("Координати:"))
        v3.addWidget(self.cb_mode)
        grp3.setLayout(v3)
        vbox.addWidget(grp3)

        # HELMERT PARAMS (Hidden by default)
        self.grp_helmert = QGroupBox("4. Параметри Гелмерта")
        self.grp_helmert.setVisible(False)
        gl = QGridLayout()

        # Dx, Dy, Dz
        self.sb_dx = self.make_spin();
        self.sb_dy = self.make_spin();
        self.sb_dz = self.make_spin()
        gl.addWidget(QLabel("Dx (m):"), 0, 0);
        gl.addWidget(self.sb_dx, 0, 1)
        gl.addWidget(QLabel("Dy (m):"), 1, 0);
        gl.addWidget(self.sb_dy, 1, 1)
        gl.addWidget(QLabel("Dz (m):"), 2, 0);
        gl.addWidget(self.sb_dz, 2, 1)

        # Rx, Ry, Rz (sec)
        self.sb_rx = self.make_spin();
        self.sb_ry = self.make_spin();
        self.sb_rz = self.make_spin()
        gl.addWidget(QLabel("Rx (\"):"), 0, 2);
        gl.addWidget(self.sb_rx, 0, 3)
        gl.addWidget(QLabel("Ry (\"):"), 1, 2);
        gl.addWidget(self.sb_ry, 1, 3)
        gl.addWidget(QLabel("Rz (\"):"), 2, 2);
        gl.addWidget(self.sb_rz, 2, 3)

        # Scale
        self.sb_s = self.make_spin();
        self.sb_s.setRange(-10000, 10000)
        gl.addWidget(QLabel("Scale (ppm):"), 3, 0);
        gl.addWidget(self.sb_s, 3, 1)

        self.grp_helmert.setLayout(gl)
        vbox.addWidget(self.grp_helmert)

        # Run Button
        self.btn_run = QPushButton("РОЗПІЗНАТИ ТА ПЕРЕТВОРИТИ")
        self.btn_run.setStyleSheet("background-color: #007acc; font-weight: bold; padding: 10px;")
        self.btn_run.clicked.connect(self.run)
        vbox.addWidget(self.btn_run)

        # Output
        self.out = QTextEdit();
        self.out.setReadOnly(True)
        vbox.addWidget(self.out)

        layout.addWidget(panel)

        # --- RIGHT PANEL ---
        r_layout = QVBoxLayout()
        # Тут треба вставити клас ImageWidget з попереднього коду (я скорочую, щоб влізло)
        self.img_wid = ImageWidget()
        r_layout.addWidget(self.img_wid, 1)
        r_layout.addWidget(QLabel("Виділіть зону мишкою для точного пошуку"))
        layout.addLayout(r_layout, 1)

    def make_spin(self):
        sb = QDoubleSpinBox()
        sb.setRange(-999999.0, 999999.0)
        sb.setDecimals(4)
        return sb

    def toggle_helmert(self):
        is_helmert = (self.cb_mode.currentText() == "Helmert (X, Y, H)")
        self.grp_helmert.setVisible(is_helmert)

    def load_img(self):
        f, _ = QFileDialog.getOpenFileName(self, "Img", "", "Images (*.jpg *.png *.bmp)")
        if f:
            img = cv2.imread(f)
            self.current_cv_image = img
            self.img_wid.set_image(img)

    def paste_img(self):
        cb = QApplication.clipboard().mimeData()
        if cb.hasImage():
            qimg = cb.imageData().convertToFormat(QImage.Format_RGB888)
            ptr = qimg.constBits()
            arr = np.array(ptr).reshape(qimg.height(), qimg.width(), 3)
            self.current_cv_image = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
            self.img_wid.set_image(self.current_cv_image)

    def run(self):
        if self.current_cv_image is None: return

        roi_rect = self.img_wid.roi_rect
        roi = (roi_rect.x(), roi_rect.y(), roi_rect.width(), roi_rect.height()) if not roi_rect.isEmpty() else (0, 0, 0,
                                                                                                                0)

        # Збираємо параметри
        h_params = {
            'dx': self.sb_dx.value(), 'dy': self.sb_dy.value(), 'dz': self.sb_dz.value(),
            'rx': self.sb_rx.value(), 'ry': self.sb_ry.value(), 'rz': self.sb_rz.value(),
            's': self.sb_s.value()
        }

        params = {
            'image': self.current_cv_image,
            'roi': roi,
            'model_mode': self.cb_model.currentText(),
            'calc_mode': self.cb_mode.currentText(),
            'lat': float(self.lat.text()),
            'lon': float(self.lon.text()),
            'helmert': h_params
        }

        self.worker = LogicWorker(params)
        self.worker.result_ready.connect(self.done)
        self.worker.start()
        self.btn_run.setEnabled(False);
        self.btn_run.setText("Processing...")

    def done(self, img, raw, gps):
        self.img_wid.set_image(img)
        self.out.setText(f"RAW:\n{raw}\n\n{gps}")
        self.btn_run.setEnabled(True);
        self.btn_run.setText("РОЗПІЗНАТИ ТА ПЕРЕТВОРИТИ")


# (Сюди вставте клас ImageWidget з попередньої відповіді без змін)
class ImageWidget(QLabel):
    def __init__(self):
        super().__init__()
        self.setAlignment(Qt.AlignCenter)
        self.setStyleSheet("background-color: #1e1e1e; border: 2px dashed #444;")
        self.setMouseTracking(True)
        self.pixmap_orig = None;
        self.start_point = None;
        self.end_point = None
        self.is_drawing = False;
        self.roi_rect = QRect()

    def set_image(self, cv_img):
        h, w, c = cv_img.shape
        q_img = QImage(cv_img.data, w, h, 3 * w, QImage.Format_RGB888).rgbSwapped()
        self.pixmap_orig = QPixmap.fromImage(q_img)
        self.setPixmap(self.pixmap_orig.scaled(self.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))
        self.roi_rect = QRect();
        self.start_point = None;
        self.end_point = None

    def mousePressEvent(self, e):
        if self.pixmap() and e.button() == Qt.LeftButton:
            self.is_drawing = True;
            self.start_point = e.pos();
            self.end_point = e.pos();
            self.update()

    def mouseMoveEvent(self, e):
        if self.is_drawing: self.end_point = e.pos(); self.update()

    def mouseReleaseEvent(self, e):
        if self.is_drawing:
            self.is_drawing = False;
            self.end_point = e.pos();
            self.calc_roi();
            self.update()

    def calc_roi(self):
        if not self.pixmap_orig: return
        scaled = self.pixmap()
        x_off = (self.width() - scaled.width()) // 2;
        y_off = (self.height() - scaled.height()) // 2
        x1 = min(self.start_point.x(), self.end_point.x()) - x_off
        y1 = min(self.start_point.y(), self.end_point.y()) - y_off
        x2 = max(self.start_point.x(), self.end_point.x()) - x_off
        y2 = max(self.start_point.y(), self.end_point.y()) - y_off
        sx = self.pixmap_orig.width() / scaled.width();
        sy = self.pixmap_orig.height() / scaled.height()
        rx = int(x1 * sx);
        ry = int(y1 * sy);
        rw = int((x2 - x1) * sx);
        rh = int((y2 - y1) * sy)
        self.roi_rect = QRect(max(0, rx), max(0, ry), min(rw, self.pixmap_orig.width() - rx),
                              min(rh, self.pixmap_orig.height() - ry))

    def paintEvent(self, e):
        super().paintEvent(e)
        if self.start_point and self.end_point:
            p = QPainter(self);
            p.setPen(QPen(QColor(0, 255, 0), 2, Qt.DashLine));
            p.setBrush(QColor(0, 255, 0, 50))
            p.drawRect(QRect(self.start_point, self.end_point).normalized())


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MainWindow()
    window.show()
    sys.exit(app.exec())

SystemExit: 0

/home/yesman/tfvenv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
import sys
import os
import cv2
import math
import re
import numpy as np
import torch
from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout,
                               QHBoxLayout, QLabel, QPushButton, QComboBox,
                               QTextEdit, QFileDialog, QGroupBox, QLineEdit,
                               QFormLayout, QFrame)
from PySide6.QtGui import QPixmap, QImage, QPainter, QPen, QColor
from PySide6.QtCore import Qt, QRect, QThread, Signal

from ultralytics import YOLO
from pyproj import CRS, Transformer, Geod

YOLO_ONLY_WEIGHTS = "runs/detect/yolo_only/weights/best.pt"
YOLO_ZONE_WEIGHTS = "runs/detect/yolo_zone_v2/weights/best.pt"
LPRNET_WEIGHTS = "weights_v2/lprnet_best.pth"
LPRNET_REPO_PATH = 'LPRNet_Pytorch'

if LPRNET_REPO_PATH not in sys.path:
    sys.path.append(LPRNET_REPO_PATH)
try:
    from model.LPRNet import LPRNet
except ImportError:
    pass

CHARS = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '.']
YOLO_NAMES = {
    0: '0', 1: '1', 2: '2', 3: '3', 4: '4',
    5: '5', 6: '6', 7: '7', 8: '8', 9: '9',
    10: 'minus', 11: 'point', 12: 'tag'
}
CHAR_MAP = {
    '0': '0', '1': '1', '2': '2', '3': '3', '4': '4',
    '5': '5', '6': '6', '7': '7', '8': '8', '9': '9',
    'minus': '-', 'point': '.', 'tag': ''
}


class LogicWorker(QThread):
    result_ready = Signal(object, str, str)

    def __init__(self, params):
        super().__init__()
        self.params = params
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def run(self):
        img_orig = self.params['image']
        roi = self.params['roi']
        mode = self.params['model_mode']

        x_off, y_off, w, h = roi
        if w > 0 and h > 0:
            img_process = np.zeros_like(img_orig)
            img_process[y_off:y_off + h, x_off:x_off + w] = img_orig[y_off:y_off + h, x_off:x_off + w]
            vis_img = img_orig.copy()
            cv2.rectangle(vis_img, (x_off, y_off), (x_off + w, y_off + h), (255, 0, 255), 2)
        else:
            img_process = img_orig
            vis_img = img_orig.copy()
        detected_texts = []

        if mode == "YOLO Only":
            model = YOLO(YOLO_ONLY_WEIGHTS)
            res = model(img_process, conf=0.25, verbose=False)[0]
            chars = []
            for box in res.boxes:
                cls = int(box.cls[0])
                sym = CHAR_MAP.get(YOLO_NAMES.get(cls, '?'), '')
                if not sym: continue
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                if w > 0 and h > 0:
                    if not (x_off <= cx <= x_off + w and y_off <= cy <= y_off + h):
                        continue

                chars.append({'char': sym, 'cx': cx, 'cy': cy, 'x': x1, 'y': y1, 'w': x2 - x1,
                              'h': y2 - y1, 'conf': float(box.conf[0])})
                cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 165, 255), 1)
            detected_texts = self.organize_text_lines_chars(chars)
        elif mode == "YOLO + LPR":
            y_zone = YOLO(YOLO_ZONE_WEIGHTS)
            try:
                lpr = LPRNet(lpr_max_len=18, phase=False, class_num=13, dropout_rate=0).to(self.device)
                lpr.load_state_dict(torch.load(LPRNET_WEIGHTS, map_location=self.device))
                lpr.eval()
            except:
                detected_texts = ["Error: LPRNet Model Not Found"]

            res = y_zone(img_process, verbose=False)[0]
            boxes = self.organize_lpr_zones(res.boxes.xyxy.cpu().numpy().astype(int).tolist())

            PAD = 5

            for b in boxes:
                x1, y1, x2, y2 = b

                if w > 0 and h > 0:
                    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                    if not (x_off <= cx <= x_off + w and y_off <= cy <= y_off + h):
                        continue

                x1 -= PAD
                y1 -= PAD
                x2 += PAD
                y2 += PAD

                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(img_orig.shape[1], x2)
                y2 = min(img_orig.shape[0], y2)

                crop = img_orig[y1:y2, x1:x2]
                if crop.size == 0: continue

                blob = cv2.resize(crop, (94, 24)).astype('float32')
                blob -= 127.5
                blob *= 0.0078125
                blob = np.transpose(blob, (2, 0, 1))
                blob = torch.from_numpy(blob).unsqueeze(0).to(self.device)

                with torch.no_grad():
                    txt = self.decode_lpr(lpr(blob))
                    detected_texts.append(txt)

                cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(vis_img, txt, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        raw_text_combined = " ".join(detected_texts)
        gps_result = self.calculate_coords(raw_text_combined)

        self.result_ready.emit(vis_img, raw_text_combined, gps_result)

    def organize_text_lines_chars(self, chars):
        if not chars: return []
        chars.sort(key=lambda c: c['conf'], reverse=True)
        valid = []
        for c in chars:
            if not any(math.hypot(c['cx'] - v['cx'], c['cy'] - v['cy']) < 5.0 for v in valid): valid.append(c)
        chars = sorted(valid, key=lambda c: c['cx'])
        lines = []
        for char in chars:
            best = None;
            mn = float('inf')
            for l in lines:
                last = l[-1]
                dx = char['cx'] - last['cx']
                dy = char['cy'] - last['cy']
                if dx <= 0: continue
                dist = math.hypot(dx, dy)
                ang = math.degrees(math.atan2(dy, dx))
                if abs(ang) > 25 or dist > (last['w'] + char['w'] + last['h'] + char['h']) / 4 * 5: continue
                if dist < mn: mn = dist; best = l
            if best:
                best.append(char)
            else:
                lines.append([char])
        lines.sort(key=lambda l: sum(c['cy'] for c in l) / len(l))
        final = []
        for l in lines:
            s = ""
            px = l[0]['x'] + l[0]['w']
            for i, c in enumerate(l):
                if i > 0 and (c['x'] - px) > (c['w'] + l[i - 1]['w']) / 2 * 0.8: s += " "
                s += c['char']
                px = c['x'] + c['w']
            final.append(s)
        return final

    def organize_lpr_zones(self, boxes):
        if not boxes: return []
        zones = [{'box': b, 'cy': (b[1] + b[3]) / 2, 'cx': (b[0] + b[2]) / 2, 'h': b[3] - b[1]} for b in boxes]
        zones.sort(key=lambda z: z['cy'])
        rows = []
        if zones:
            curr = [zones[0]]
            for i in range(1, len(zones)):
                c = zones[i]
                p = curr[-1]
                if abs(c['cy'] - p['cy']) < p['h'] * 0.6:
                    curr.append(c)
                else:
                    rows.append(curr); curr = [c]
            rows.append(curr)
        final = []
        for r in rows:
            r.sort(key=lambda z: z['cx'])
            for item in r: final.append(item['box'])
        return final

    def decode_lpr(self, preds):
        preds = preds.cpu().detach().numpy()
        labels = np.argmax(preds, axis=1)
        decoded = []
        for i in range(labels.shape[0]):
            line = labels[i, :]
            res = []
            pre = -1
            for c in line:
                if c != pre and c != 12: res.append(CHARS[c])
                pre = c
            decoded.append("".join(res))
        return decoded[0]

    def calculate_coords(self, text):
        nums = re.findall(r'-?\d+\.?\d*', text)
        nums = [float(n) for n in nums]

        calc_mode = self.params['calc_mode']

        r_lat = self.params['lat']
        r_lon = self.params['lon']
        r_alt = self.params['alt']

        if calc_mode == "Polar (Az, Dist)":
            if len(nums) >= 2:
                az, dist = nums[0], nums[1]
                geod = Geod(ellps='WGS84')
                lon_end, lat_end, _ = geod.fwd(r_lon, r_lat, az, dist)
                return f"MODE: Polar\nAz={az}, Dist={dist}\n\nGPS RESULT:\nLat: {lat_end:.6f}\nLon: {lon_end:.6f}"
            else:
                return "Error: Need Az, Dist"

        elif calc_mode == "Local ENU (X, Y, H)":
            if len(nums) >= 2:
                x, y = nums[0], nums[1]
                z = nums[2] if len(nums) > 2 else 0.0

                try:
                    pipeline_str = (
                        f"+proj=pipeline "
                        f"+step +proj=topocentric +ellps=WGS84 +lat_0={r_lat} +lon_0={r_lon} +h_0={r_alt} +inv "
                        f"+step +proj=cart +ellps=WGS84 +inv "
                        f"+step +proj=unitconvert +xy_in=rad +xy_out=deg"
                    )

                    transformer = Transformer.from_pipeline(pipeline_str)
                    res_lon, res_lat, res_alt = transformer.transform(x, y, z)
                    simple_alt = r_alt + z
                    diff = res_alt - simple_alt

                    return (f"MODE: ENU -> WGS84 (Topocentric)\n"
                            f"Input: X={x}, Y={y}, Z={z}\n"
                            f"Radar Alt: {r_alt} m\n\n"
                            f"GPS RESULT:\n"
                            f"Lat: {res_lat:.8f}\n"
                            f"Lon: {res_lon:.8f}\n"
                            f"Alt: {res_alt:.3f} m\n"
                            f"(Curvature correct: {diff:+.3f}m vs flat model)")

                except Exception as e:
                    return f"Projection Error: {e}"
            else:
                return "Error: Need at least X, Y"

        return "Unknown Mode"

class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Radar Digit Recognizer v2.0 (ENU Support)")
        self.resize(1250, 650)
        self.current_cv_image = None

        self.setStyleSheet("""
            QMainWindow { background-color: #2b2b2b; color: white; }
            QLabel { color: #e0e0e0; font-size: 14px; }
            QGroupBox { border: 1px solid #555; margin-top: 10px; font-weight: bold; color: #00bcd4; }
            QPushButton { background-color: #3d3d3d; border: 1px solid #555; padding: 6px; border-radius: 4px; color: white; }

            QPushButton:hover { background-color: #505050; border-color: #00bcd4; }
            QLineEdit, QComboBox, QTextEdit { background-color: #1e1e1e; border: 1px solid #555; color: white; padding: 4px; }
        """)

        main_widget = QWidget()
        self.setCentralWidget(main_widget)
        layout = QHBoxLayout(main_widget)

        panel = QFrame();
        panel.setFixedWidth(400)
        vbox = QVBoxLayout(panel)

        grp1 = QGroupBox("1. Зображення")
        l1 = QHBoxLayout()
        btn1 = QPushButton("Файл")
        btn1.clicked.connect(self.load_img)
        btn2 = QPushButton("Буфер")
        btn2.clicked.connect(self.paste_img)
        l1.addWidget(btn1)
        l1.addWidget(btn2)
        grp1.setLayout(l1)
        vbox.addWidget(grp1)

        grp2 = QGroupBox("2. Позиція Радара (Origin)")
        f2 = QFormLayout()
        self.lat = QLineEdit("50.0")
        self.lon = QLineEdit("50.0")
        self.alt = QLineEdit("200.0")
        f2.addRow("Lat (deg):", self.lat)
        f2.addRow("Lon (deg):", self.lon)
        f2.addRow("Alt (m):", self.alt)
        grp2.setLayout(f2)
        vbox.addWidget(grp2)

        grp3 = QGroupBox("3. Налаштування")
        v3 = QVBoxLayout()
        self.cb_model = QComboBox()
        self.cb_model.addItems(["YOLO Only", "YOLO + LPR"])
        self.cb_model.activated.connect(lambda: self.closeComboPopup(self.cb_model))

        v3.addWidget(QLabel("Модель:"))
        v3.addWidget(self.cb_model)

        self.cb_mode = QComboBox()
        self.cb_mode.addItems(["Local ENU (X, Y, H)", "Polar (Az, Dist)"])
        v3.addWidget(QLabel("Система координат:"))
        v3.addWidget(self.cb_mode)
        grp3.setLayout(v3)
        vbox.addWidget(grp3)

        self.btn_run = QPushButton("РОЗПІЗНАТИ")
        self.btn_run.setStyleSheet("background-color: #007acc; font-weight: bold; padding: 12px; font-size: 14px;")
        self.btn_run.clicked.connect(self.run)
        vbox.addWidget(self.btn_run)

        self.out = QTextEdit()
        self.out.setReadOnly(True)
        self.out.setPlaceholderText("Результат буде тут...")
        vbox.addWidget(self.out)

        layout.addWidget(panel)

        r_layout = QVBoxLayout()
        self.img_wid = ImageWidget()
        r_layout.addWidget(self.img_wid, 1)
        r_layout.addWidget(QLabel("Підказка: Виділіть зону з цифрами мишкою для точності"))
        layout.addLayout(r_layout, 1)

    def closeComboPopup(self, combo: QComboBox):
        combo.hidePopup()
        combo.repaint()
        combo.update()
        self.repaint()
        self.update()

    def load_img(self):
        f, _ = QFileDialog.getOpenFileName(self, "Img", "", "Images (*.jpg *.png *.bmp)")
        if f:
            img = cv2.imread(f)
            self.current_cv_image = img
            self.img_wid.set_image(img)

    def paste_img(self):
        cb = QApplication.clipboard().mimeData()
        if cb.hasImage():
            qimg = cb.imageData().convertToFormat(QImage.Format_RGB888)
            ptr = qimg.constBits()
            arr = np.array(ptr).reshape(qimg.height(), qimg.width(), 3)
            self.current_cv_image = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
            self.img_wid.set_image(self.current_cv_image)

    def run(self):
        if self.current_cv_image is None: return
        roi_rect = self.img_wid.roi_rect
        roi = (roi_rect.x(), roi_rect.y(), roi_rect.width(), roi_rect.height()) if not roi_rect.isEmpty() else (0, 0, 0, 0)

        params = {
            'image': self.current_cv_image,
            'roi': roi,
            'model_mode': self.cb_model.currentText(),
            'calc_mode': self.cb_mode.currentText(),
            'lat': float(self.lat.text()),
            'lon': float(self.lon.text()),
            'alt': float(self.alt.text())  # Передаємо висоту
        }

        self.worker = LogicWorker(params)
        self.worker.result_ready.connect(self.done)
        self.worker.start()
        self.btn_run.setEnabled(False)
        self.btn_run.setText("Обробка...")

    def done(self, img, raw, gps):
        self.img_wid.set_image(img)
        self.out.setText(f"--- RAW DATA ---\n{raw}\n\n--- GEODETIC ---\n{gps}")
        self.btn_run.setEnabled(True)
        self.btn_run.setText("РОЗПІЗНАТИ")


class ImageWidget(QLabel):
    def __init__(self):
        super().__init__()
        self.setAlignment(Qt.AlignCenter)
        self.setStyleSheet("background-color: #1e1e1e; border: 2px dashed #444;")
        self.setMouseTracking(True)
        self.pixmap_orig = None
        self.start_point = None
        self.end_point = None
        self.is_drawing = False
        self.roi_rect = QRect()

    def set_image(self, cv_img):
        h, w, c = cv_img.shape
        q_img = QImage(cv_img.data, w, h, 3 * w, QImage.Format_RGB888).rgbSwapped()
        self.pixmap_orig = QPixmap.fromImage(q_img)
        self.setPixmap(self.pixmap_orig.scaled(self.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))
        self.roi_rect = QRect();
        self.start_point = None;
        self.end_point = None

    def mousePressEvent(self, e):
        if self.pixmap() and e.button() == Qt.LeftButton:
            self.is_drawing = True
            self.start_point = e.pos()
            self.end_point = e.pos()
            self.update()

    def mouseMoveEvent(self, e):
        if self.is_drawing: self.end_point = e.pos(); self.update()

    def mouseReleaseEvent(self, e):
        if self.is_drawing:
            self.is_drawing = False
            self.end_point = e.pos()
            self.calc_roi()
            self.update()

    def calc_roi(self):
        if not self.pixmap_orig: return
        scaled = self.pixmap()
        x_off = (self.width() - scaled.width()) // 2
        y_off = (self.height() - scaled.height()) // 2
        x1 = min(self.start_point.x(), self.end_point.x()) - x_off
        y1 = min(self.start_point.y(), self.end_point.y()) - y_off
        x2 = max(self.start_point.x(), self.end_point.x()) - x_off
        y2 = max(self.start_point.y(), self.end_point.y()) - y_off
        sx = self.pixmap_orig.width() / scaled.width()
        sy = self.pixmap_orig.height() / scaled.height()
        rx = int(x1 * sx)
        ry = int(y1 * sy)
        rw = int((x2 - x1) * sx)
        rh = int((y2 - y1) * sy)
        self.roi_rect = QRect(max(0, rx), max(0, ry), min(rw, self.pixmap_orig.width() - rx),
                              min(rh, self.pixmap_orig.height() - ry))

    def paintEvent(self, e):
        super().paintEvent(e)
        if self.start_point and self.end_point:
            p = QPainter(self)
            p.setPen(QPen(QColor(0, 255, 0), 2, Qt.DashLine))
            p.setBrush(QColor(0, 255, 0, 50))
            p.drawRect(QRect(self.start_point, self.end_point).normalized())


if __name__ == "__main__":
    os.environ["QT_AUTO_SCREEN_SCALE_FACTOR"] = "1"
    os.environ["QT_SCALE_FACTOR"] = "1.5"
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    else:
        print("Використовується вже існуючий екземпляр QApplication")
    window = MainWindow()
    window.show()
    app.exec()

Використовується вже існуючий екземпляр QApplication


/tmp/ipykernel_26334/4270851070.py:418: DeprecationWarning: Function: 'QMouseEvent.pos() const' is marked as deprecated, please check the documentation for more information.
  self.start_point = e.pos()
/tmp/ipykernel_26334/4270851070.py:419: DeprecationWarning: Function: 'QMouseEvent.pos() const' is marked as deprecated, please check the documentation for more information.
  self.end_point = e.pos()
/tmp/ipykernel_26334/4270851070.py:423: DeprecationWarning: Function: 'QMouseEvent.pos() const' is marked as deprecated, please check the documentation for more information.
  if self.is_drawing: self.end_point = e.pos(); self.update()
/tmp/ipykernel_26334/4270851070.py:428: DeprecationWarning: Function: 'QMouseEvent.pos() const' is marked as deprecated, please check the documentation for more information.
  self.end_point = e.pos()
